# 경구약제 이미지 인식 기능을 이용한 유저 헬스케어 데이터 수집 서비스_EDA



## EDA 목적

데이터 구조와 특성을 근거로 학습에 필요한 전처리, 검증 데이터 분할, 추가 데이터 확보 방향을 결정한다.

## 목차

1. **데이터 파일 구성 이해**

   폴더 위치·역할·하위 구조, 파일 형식·개수·용량, 파일명 형태를 확인한다. 원본 파일의 탐색·로딩 기준과 추가 확인할 파일명 규칙을 정리한다.

   - **1.1 데이터 위치와 폴더 구조**

     - 1.1.1 기본 폴더의 위치와 역할
     - 1.1.2 라벨 폴더의 하위 구조
   - **1.2 파일 형식·개수·용량**

   - **1.3 파일명 형태와 규칙 확인**

   - **1.4 학습 준비를 위한 결론과 후속 확인**


2. **라벨 구조와 학습용 정답 구성 이해**

   JSON 구조를 요약하고 이미지·연결, 약품·클래스, 외형·식별, 촬영, 객체·박스 정보로 나누어 확인한다. 각 용도별로 필드 정의·실제 값·자료형과 관련 검증을 함께 살펴보고 학습 준비에 필요한 조치를 정리한다.

   - **2.1 JSON 전체 구조 요약**

     - 2.1.1 샘플 JSON의 최상위 구조
     - 2.1.2 샘플의 항목 개수와 유형
     - 2.1.3 전체 JSON의 구성과 집계 단위
     - 2.1.4 필드 정의와 실제 값의 집계 기준
     - 2.1.5 분석 결과와 필요한 조치
   - **2.2 이미지 정보와 라벨 연결**

     - 2.2.1 이미지 필드의 정의·실제 값·자료형
     - 2.2.2 이미지 크기 필드의 제공 범위와 일관성
     - 2.2.3 실제 이미지 파일과 JSON의 참조 관계
     - 2.2.4 JSON 내부의 이미지 ID 연결과 전체 ID 범위
     - 2.2.5 이미지별 실제 연결 예시
     - 2.2.6 파일명 코드와 JSON 이미지 코드 비교
     - 2.2.7 분석 결과와 필요한 조치
   - **2.3 약품·클래스 정보**

     - 2.3.1 약품·클래스 필드의 정의·실제 값·자료형
     - 2.3.2 클래스 목록과 원본 ID의 일관성
     - 2.3.3 객체별 클래스 참조와 약품 정보의 대응
     - 2.3.4 약품 코드·클래스·약품명 대응표
     - 2.3.5 분석 결과와 필요한 조치
   - **2.4 약제 외형·식별정보**

     - 2.4.1 외형·식별 필드의 정의·실제 값·자료형
     - 2.4.2 분석 결과와 필요한 조치
   - **2.5 촬영정보**

     - 2.5.1 촬영 필드의 정의·실제 값·자료형
     - 2.5.2 촬영 필드의 제공 범위와 일관성
     - 2.5.3 분석 결과와 필요한 조치
   - **2.6 객체 정답·박스 정보**

     - 2.6.1 객체·박스 필드의 정의·실제 값·자료형
     - 2.6.2 분석 결과와 필요한 조치
   - **2.7 학습 준비를 위한 결론과 후속 확인**


3. **샘플 이미지 및 정답 박스 시각화**

   실제 객체와 박스의 대응, 배경·조명·촬영 방향, 객체의 겹침 등을 관찰한다. 라벨의 정확성을 확인하고, 필요한 전처리와 데이터 증강 후보를 정리한다.


4. **데이터 분포 분석**

   클래스별 이미지·객체 수, 이미지 크기, 객체 크기, 이미지당 객체 수를 분석한다. 클래스 불균형 대응, 입력 이미지 크기, 크기 조정 및 샘플링 방식을 검토한다.


5. **데이터 품질 검증**

   누락된 라벨, 중복 데이터, 손상된 이미지, 잘못된 박스 좌표를 확인한다. 학습 전 데이터 정제와 오류 처리 기준을 정리한다.


6. **검증 데이터 분할 및 추가 데이터 전략**

   촬영 장면·유사 이미지의 관계와 클래스 분포를 고려해 학습·검증 분할 기준을 정한다. 부족한 클래스나 촬영 조건을 파악하고, 외부 데이터의 필요성과 필요한 데이터의 조건을 판단한다.


각 분석은 **확인할 질문 → 근거 → 학습에 반영할 결정 또는 후속 실험**으로 정리한다.


---

## 1. 데이터 파일 구성 이해

**목적:** 원본 데이터의 위치·계층·파일 형식·규모·파일명 형태를 파악하고,
학습에 사용할 파일을 탐색하고 읽는 기준을 정한다.

### 1.1 데이터 위치와 폴더 구조

**목적:** 학습 이미지·라벨·테스트 이미지의 위치와 역할을 확인하고,
라벨 파일이 저장된 하위 구조를 파악한다.

#### 1.1.1 기본 폴더의 위치와 역할


In [55]:
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
import json
from IPython.display import Markdown, display

# 저장소 루트 또는 notebooks 폴더에서 실행할 수 있도록 경로를 찾는다.
cwd = Path.cwd().resolve()
project_root = next(
    (p for p in [cwd, *cwd.parents] if (p / "data/raw").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("프로젝트의 data/raw 폴더를 찾지 못했습니다.")

data_dir = project_root / "data/raw"
folder_roles = {
    "train_images": "학습 이미지",
    "train_annotations": "학습 정답 라벨",
    "test_images": "테스트 이미지",
}

def show_table(headers, rows):
    # 파일명·약품명에 포함될 수 있는 표 구분 문자와 줄바꿈을 처리한다.
    def text(value):
        return str(value).replace("|", "&#124;").replace("\n", " ")
    lines = [
        "| " + " | ".join(map(text, headers)) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    lines.extend("| " + " | ".join(map(text, row)) + " |" for row in rows)
    display(Markdown("\n".join(lines)))

missing_folders = [
    name for name in folder_roles if not (data_dir / name).is_dir()
]
if missing_folders:
    raise FileNotFoundError(f"필수 폴더가 없습니다: {missing_folders}")

show_table(
    ["역할", "프로젝트 기준 경로", "존재 여부"],
    [
        [role, f"data/raw/{name}", "확인"]
        for name, role in folder_roles.items()
    ],
)
raw_subfolders = sorted(
    p.name for p in data_dir.iterdir()
    if p.is_dir() and not p.name.startswith(".")
)
display(Markdown("**data/raw의 하위 폴더:** " + ", ".join(raw_subfolders)))

| 역할 | 프로젝트 기준 경로 | 존재 여부 |
| --- | --- | --- |
| 학습 이미지 | data/raw/train_images | 확인 |
| 학습 정답 라벨 | data/raw/train_annotations | 확인 |
| 테스트 이미지 | data/raw/test_images | 확인 |

**data/raw의 하위 폴더:** test_images, train_annotations, train_images

- **관찰한 사실:** data/raw에는 학습 이미지·학습 라벨·테스트 이미지 폴더가 있으며, 별도 검증 폴더는 없다.
- **학습에 미치는 영향:** 각 폴더의 역할을 구분하고, 라벨이 있는 데이터에서 검증셋을 구성할 기준이 필요하다.
- **필요한 조치:** 이미지·라벨 경로를 분리해 관리하고, 검증 분할은 이미지 간 관계와 클래스 분포를 고려한다.

**후속 확인:** **6번**에서 유사 촬영 이미지와 클래스 분포를 고려한 분할 기준을 정한다.

#### 1.1.2 라벨 폴더의 하위 구조

**목적:** _json 항목의 실제 종류와 내부 계층을 확인해 JSON 파일의 탐색 방법을 정한다.

In [56]:
from pathlib import Path
from itertools import islice

annotation_dir = data_dir / "train_annotations"

for item in islice(annotation_dir.iterdir(), 3):
    print(f"\n{item.name}")
    print("종류:", "폴더" if item.is_dir() else "파일")

    if item.is_dir():
        for child in islice(item.iterdir(), 3):
            print("  └", child.name)


K-003351-016232-033880_json
종류: 폴더
  └ K-003351
  └ K-016232
  └ K-033880

K-003351-035206-041768_json
종류: 폴더
  └ K-003351
  └ K-041768
  └ K-035206

K-003351-013900-029667_json
종류: 폴더
  └ K-003351
  └ K-029667
  └ K-013900


- **관찰한 사실:** 샘플의 _json 항목은 파일이 아니라 폴더이며, 내부에 K-코드로 된 하위 항목이 있다.
- **학습에 미치는 영향:** 최상위 항목만 읽으면 실제 라벨 파일을 놓칠 수 있다.
- **필요한 조치:** 하위 폴더까지 재귀 탐색하여 .json 파일을 읽고, 폴더 수와 파일 수를 구분한다.

### 1.2 파일 형식·개수·용량

**목적:** 전체 하위 파일의 확장자·개수·용량을 확인해 학습 데이터 규모와 저장 공간을 파악한다.

In [57]:
files_by_folder = {}
inventory_rows = []
zero_byte_paths = []
total_bytes = 0

for folder_name in folder_roles:
    folder = data_dir / folder_name
    files = sorted(
        p for p in folder.rglob("*")
        if p.is_file()
        and not any(
            part.startswith(".") for part in p.relative_to(folder).parts
        )
    )
    files_by_folder[folder_name] = files
    extension_counts = Counter(
        p.suffix.lower() or "(확장자 없음)" for p in files
    )
    sizes = {p: p.stat().st_size for p in files}
    folder_bytes = sum(sizes.values())
    total_bytes += folder_bytes
    empty_files = [p for p, size in sizes.items() if size == 0]
    zero_byte_paths.extend(empty_files)
    inventory_rows.append([
        folder_name,
        len(files),
        ", ".join(f"{ext}: {count}" for ext, count in sorted(extension_counts.items())),
        f"{folder_bytes / 1024**2:,.2f}",
        len(empty_files),
    ])

show_table(
    ["폴더", "파일 수", "확장자별 개수", "용량(MiB)", "0바이트 파일 수"],
    inventory_rows,
)
display(Markdown(
    f"**전체 원본 파일 용량:** {total_bytes / 1024**3:,.3f} GiB "
    "(MiB와 GiB는 각각 1024의 거듭제곱 기준)"
))

image_suffixes = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}
train_image_paths = [
    p for p in files_by_folder["train_images"]
    if p.suffix.lower() in image_suffixes
]
test_image_paths = [
    p for p in files_by_folder["test_images"]
    if p.suffix.lower() in image_suffixes
]
json_paths = [
    p for p in files_by_folder["train_annotations"]
    if p.suffix.lower() == ".json"
]

| 폴더 | 파일 수 | 확장자별 개수 | 용량(MiB) | 0바이트 파일 수 |
| --- | --- | --- | --- | --- |
| train_images | 232 | .png: 232 | 387.67 | 0 |
| train_annotations | 763 | .json: 763 | 1.98 | 0 |
| test_images | 842 | .png: 842 | 1,446.55 | 0 |

**전체 원본 파일 용량:** 1.793 GiB (MiB와 GiB는 각각 1024의 거듭제곱 기준)

- **관찰한 사실:** 학습 이미지 232개·테스트 이미지 842개는 PNG, 라벨 파일 763개는 JSON이다. 총 용량은 약 1.793 GiB이며 0바이트 파일은 없다.
- **학습에 미치는 영향:** 이미지 파일 수와 라벨 파일 수가 다르므로 JSON 개수를 학습 이미지 수로 해석할 수 없다.
- **필요한 조치:** 이미지와 라벨의 참조 관계를 확인하고, 전처리 결과와 모델 결과를 저장할 공간을 준비한다.

**집계 기준:** 숨김 파일은 제외했다. 0바이트가 없다는 사실만으로 이미지가 정상 디코딩된다고 판단하지 않는다.

**후속 확인:** **2.1~2.2**에서 라벨 집계와 이미지 참조 관계를 확인하고, **5번**에서 손상 파일을 검사한다.

### 1.3 파일명 형태와 규칙 확인

**목적:** 폴더별 항목을 최대 3개씩 출력해 파일명 형태를 관찰하고,
이름을 해석하기 위해 라벨 정보와 대조할 사항을 정리한다.

In [58]:
from pathlib import Path
from itertools import islice

data_dir = project_root / "data/raw"

for folder_name in ["train_images", "train_annotations", "test_images"]:
    folder = data_dir / folder_name
    print(f"\n[{folder_name}]")
    for item in islice(folder.iterdir(), 3):
        print(item.name)


[train_images]
K-003483-025367-027733-035206_0_2_0_2_75_000_200.png
K-003483-025469-030308-035206_0_2_0_2_90_000_200.png
K-003351-013900-022074_0_2_0_2_75_000_200.png

[train_annotations]
K-003351-016232-033880_json
K-003351-035206-041768_json
K-003351-013900-029667_json

[test_images]
1186.png
412.png
374.png


- **관찰한 사실:** 학습 이미지 샘플은 K-코드와 숫자들이 결합된 이름이며, 테스트 이미지 샘플은 숫자 이름이다.
- **학습에 미치는 영향:** 학습·테스트 이름에 같은 의미가 담겼다고 가정하거나 파일명만으로 클래스를 정하면 잘못된 정답을 만들 수 있다.
- **필요한 조치:** K-코드와 뒤쪽 숫자가 라벨의 약품 ID·촬영 정보와 대응하는지 확인한다.

**후속 확인:** **2.2·2.3·2.5**의 이미지 참조·클래스·촬영 정보를 근거로 파일명 구성 요소를 대조한다. 숫자의 의미와 단위는 아직 확정하지 않았다.

### 1.4 학습 준비를 위한 결론과 후속 확인

| 확인한 근거 | 학습 준비에 필요한 사항 | 이어서 확인할 단계 |
| --- | --- | --- |
| 이미지·라벨·테스트 폴더가 구분되고 라벨은 중첩 폴더에 저장됨 | 역할별 경로를 구분하고 하위 JSON을 재귀 탐색 | 2.1: 전체 JSON 읽기 |
| 학습 이미지 232개와 라벨 JSON 763개의 수가 다름 | JSON 수와 이미지 수를 구분하고 참조 관계 확인 | 2.1~2.2: 집계와 이미지별 연결 확인 |
| 학습·테스트 파일명 형태가 다름 | 파일명을 임의로 해석하지 않고 라벨 정보와 대조 | 2.2·2.3·2.5: 이미지·약품 ID·촬영 필드와 대응 확인 |
| 별도 검증 폴더가 없음 | 라벨이 있는 데이터에서 검증셋 구성 기준 필요 | 6번: 클래스 분포·촬영 그룹을 고려한 분할 |
| 원본 약 1.793 GiB, 0바이트 파일 없음 | 추가 저장 공간 확보와 실제 이미지 읽기 검사 필요 | 5번: 손상 파일 검사 |

---

## 2. 라벨 구조와 학습용 정답 구성 이해

**목적:** JSON의 필드를 용도별로 이해하고 실제 값과 관련 검증 결과를 근거로
이미지별 정답 구성, 클래스 관리, 전처리와 검증 분할에 필요한 기준을 정한다.

### 2.1 JSON 전체 구조 요약

**목적:** JSON의 최상위 구조와 전체 파일의 구성을 확인하고,
파일 수·이미지 정보 수·객체 정답 수·클래스 정의 수를 구분한다.

JSON 객체는 Python에서 `dict`로 읽으며 `key: value` 쌍으로 구성된다.
`images`·`annotations`·`categories`의 값은 `list`이고, 각 목록의 항목은 다시 `dict`이다.
따라서 `images.id`와 `annotations.id`는 서로 다른 구조에 속한 key이다.

#### 2.1.1 샘플 JSON의 최상위 구조


In [59]:
import json
from pathlib import Path

annotation_dir = data_dir / "train_annotations"
json_path = next(annotation_dir.rglob("*.json"), None)

if json_path is None:
    print("JSON 파일을 찾지 못했어요.")
else:
    print("확인할 파일:", json_path.relative_to(annotation_dir))

    with json_path.open(encoding="utf-8-sig") as f:
        label = json.load(f)

    print("최상위 자료형:", type(label).__name__)

    if isinstance(label, dict):
        for key, value in label.items():
            print(f"- {key}: {type(value).__name__}")
    elif isinstance(label, list):
        print("항목 수:", len(label))
        if label and isinstance(label[0], dict):
            print("첫 항목의 키:", list(label[0].keys()))

확인할 파일: K-003351-016232-033880_json/K-003351/K-003351-016232-033880_0_2_0_2_75_000_200.json
최상위 자료형: dict
- images: list
- type: str
- annotations: list
- categories: list


#### 2.1.2 샘플의 항목 개수와 유형

**목적:** 샘플 하나의 이미지·객체 정답·클래스 정의 항목 수를 확인하고 전체 집계와 구분한다.

In [60]:
print("이미지 정보 개수:", len(label["images"]))
print("라벨 개수:", len(label["annotations"]))
print("클래스 정보 개수:", len(label["categories"]))
print("type의 실제 값:", label["type"])

이미지 정보 개수: 1
라벨 개수: 1
클래스 정보 개수: 1
type의 실제 값: instances


#### 2.1.3 전체 JSON의 구성과 집계 단위

**목적:** 샘플에서 본 구조가 전체 JSON에도 적용되는지 확인하고 읽기 실패와 구조 차이를 점검한다.

In [61]:
image_rows = []
annotation_rows = []
category_rows = []
json_summary = []
json_errors = []
structure_errors = []

for json_path in json_paths:
    relative_path = str(json_path.relative_to(data_dir))
    try:
        with json_path.open(encoding="utf-8-sig") as f:
            document = json.load(f)
    except (OSError, UnicodeError, json.JSONDecodeError) as error:
        json_errors.append([relative_path, str(error)])
        continue

    if not isinstance(document, dict):
        structure_errors.append([relative_path, "JSON 최상위가 dict가 아님"])
        continue

    blocks = {}
    for key in ["images", "annotations", "categories"]:
        value = document.get(key)
        if not isinstance(value, list):
            structure_errors.append([relative_path, f"{key}가 list가 아님"])
            value = []
        if any(not isinstance(item, dict) for item in value):
            structure_errors.append([relative_path, f"{key}에 dict 외 항목이 있음"])
        blocks[key] = [item for item in value if isinstance(item, dict)]

    json_summary.append({
        "json_path": relative_path,
        "images": len(blocks["images"]),
        "annotations": len(blocks["annotations"]),
        "categories": len(blocks["categories"]),
    })
    for image in blocks["images"]:
        file_name = image.get("file_name")
        image_name = (
            PurePosixPath(file_name.replace("\\", "/")).name
            if isinstance(file_name, str) and file_name else None
        )
        image_rows.append({
            "json_path": relative_path,
            "image_name": image_name,
            "metadata": image,
        })
    annotation_rows.extend(
        {"json_path": relative_path, **item}
        for item in blocks["annotations"]
    )
    category_rows.extend(
        {"json_path": relative_path, **item}
        for item in blocks["categories"]
    )

show_table(
    ["집계 항목", "개수"],
    [
        ["JSON 파일", len(json_paths)],
        ["최상위 구조를 확인한 JSON", len(json_summary)],
        ["JSON 읽기 실패", len(json_errors)],
        ["예상 구조와 다른 항목", len(structure_errors)],
        ["images 항목 합계", len(image_rows)],
        ["annotations 원시 항목 합계", len(annotation_rows)],
        ["categories 항목 합계", len(category_rows)],
    ],
)
json_shapes = Counter(
    (row["images"], row["annotations"], row["categories"])
    for row in json_summary
)
show_table(
    ["JSON당 images", "JSON당 annotations", "JSON당 categories", "해당 JSON 수"],
    [[*shape, count] for shape, count in sorted(json_shapes.items())],
)
if json_errors or structure_errors:
    show_table(["파일", "확인 필요 사항"], (json_errors + structure_errors)[:10])

| 집계 항목 | 개수 |
| --- | --- |
| JSON 파일 | 763 |
| 최상위 구조를 확인한 JSON | 763 |
| JSON 읽기 실패 | 0 |
| 예상 구조와 다른 항목 | 0 |
| images 항목 합계 | 763 |
| annotations 원시 항목 합계 | 763 |
| categories 항목 합계 | 763 |

| JSON당 images | JSON당 annotations | JSON당 categories | 해당 JSON 수 |
| --- | --- | --- | --- |
| 1 | 1 | 1 | 763 |

#### 2.1.4 필드 정의와 실제 값의 집계 기준

**정의 출처:** [AI-Hub 경구약제 이미지 데이터의 JSON 구조표](https://www.aihub.or.kr/aihubdata/data/view.do?aihubDataSe=data&currMenu=120&dataSetSn=576&topMenu=100).

**정의 기재 원칙:** 정의 칸은 위 사이트 JSON 구조표의 `설명` 열 문구만 사용한다. 표 안의 줄바꿈은 공백으로 정리한다.

`—`는 해당 key가 사이트 구조표에 없어 설명을 기재하지 않았다는 표시이며, 현재 데이터에서는 `images.imgfile`이 해당한다.

값·자료형의 집계 결과와 단위·좌표·활용에 관한 검토는 공식 정의와 구분한다.


**집계 기준:** 현재 학습 라벨 JSON 전체를 읽어 key별 값을 집계한다.

images 항목에는 같은 이미지가 반복될 수 있으며, 항목 수와 고유 이미지 수는 구분한다.

표의 key에는 소속 구조를 붙여 `images.id`처럼 표시한다.

- 값이 하나면 동일 값, 고유 값이 10개 이하면 실제 값을 모두 표시한다.
- 고유 숫자가 많으면 관찰 최솟값~최댓값과 고유 값 수를 표시한다.
- 문자열 종류가 많으면 실제 예시 1개와 고유 값 수를 표시하며 예시 자체는 자르지 않는다.
- 숫자 목록은 실제 예시·길이·원소별 관찰 범위를 표시한다.
- null, 빈 문자열, key 누락을 구분한다. JSON의 null은 Python에서 None이며 자료형은 NoneType이다.

**해석 기준:** 범위는 현재 데이터의 관찰값이며 정상·허용 범위가 아니다.

동일 값·범위는 null·빈 문자열을 제외한 값 기준이고, 자료형은 null을 포함한 전체 값 기준이다.

식별자 범위는 번호의 최솟값·최댓값으로, 중간 번호가 모두 존재한다는 뜻이 아니다.

In [76]:
# AI-Hub JSON 구조표의 '설명' 열만 기재한다. 표 안의 줄바꿈은 공백으로 표시한다.
# 출처: https://www.aihub.or.kr/aihubdata/data/view.do?aihubDataSe=data&currMenu=120&dataSetSn=576&topMenu=100
# 해석·주의 사항은 정의와 구분하여 마크다운에 기록한다.
field_definitions = {
    "images": {
        "id": "약제 이미지식별자",
        "file_name": "약제 이미지파일명",
        "width": "약제 이미지너비",
        "height": "약제 이미지높이",
        "drug_N": "알약코드",
        "drug_S": "알약상태",
        "back_color": "촬영배경",
        "drug_dir": "알약방향",
        "light_color": "촬영조명",
        "camera_la": "카메라위도",
        "camera_lo": "카메라경도",
        "size": "이미지 사이즈",
        "dl_idx": "알약 식별자",
        "dl_mapping_code": "제품코드",
        "dl_name": "제품명",
        "dl_name_en": "제품명(영어)",
        "img_key": "이미지 링크",
        "dl_material": "성분명",
        "dl_material_en": "성분명(영어)",
        "dl_custom_shape": "제조 모양",
        "dl_company": "제조사명",
        "dl_company_en": "제조사명(영어)",
        "di_company_mf": "위탁제조사명",
        "di_company_mf_en": "위탁제조사명(영어)",
        "item_seq": "품목기준코드",
        "di_item_permit_date": "허가일자",
        "di_class_no": "약품 분류",
        "di_etc_otc_code": "전문의약품/ 일반의약품",
        "di_edi_code": "EDI 코드",
        "chart": "알약 제형",
        "drug_shape": "알약 모양",
        "thick": "알약 두께",
        "leng_long": "알약 장축",
        "leng_short": "알약 단축",
        "print_front": "식별문자_앞",
        "print_back": "식별문자_뒤",
        "color_class1": "색상 1",
        "color_class2": "색상 2",
        "line_front": "알약 앞면 분할선 여부",
        "line_back": "알약 뒷면 분할선 여부",
        "img_regist_ts": "약학정보원 이미지 생성일",
        "form_code_name": "정제 분류명",
        "mark_code_front_anal": "알약 앞면 마크 형태",
        "mark_code_back_anal": "알약 뒷면 마크 형태",
        "mark_code_front_img": "알약 앞면 마크 이미지",
        "mark_code_back_img": "알약 뒷면 마크 이미지",
        "mark_code_front": "알약 앞면 마크 코드",
        "mark_code_back": "알약 뒷면 마크 코드",
        "change_date": "변경일자"
    },
    "annotations": {
        "area": "바운딩박스 면적",
        "iscrowd": "평가 분류",
        "bbox": "bbox 좌표",
        "category_id": "category 아이디",
        "ignore": "무시 여부",
        "segmentation": "라벨링 바운딩 박스",
        "id": "어노테이션 아이디",
        "image_id": "이미지 아이디"
    },
    "categories": {
        "supercategory": "슈퍼 카테고리",
        "id": "카테고리 아이디",
        "name": "카테고리 명"
    }
}

# 파일 하나의 값이 아니라 전체 JSON의 값을 모은다.
field_records = {section: [] for section in field_definitions}
field_paths = sorted((data_dir / "train_annotations").rglob("*.json"))
field_read_errors = []
field_structure_errors = []
for field_path in field_paths:
    try:
        with field_path.open(encoding="utf-8-sig") as field_file:
            field_document = json.load(field_file)
    except (OSError, UnicodeError, json.JSONDecodeError) as error:
        field_read_errors.append([str(field_path), str(error)])
        continue
    if not isinstance(field_document, dict):
        field_structure_errors.append([str(field_path), "최상위가 dict가 아님"])
        continue
    for section in field_definitions:
        items = field_document.get(section)
        if not isinstance(items, list):
            field_structure_errors.append([str(field_path), section, "list가 아님"])
            continue
        for item in items:
            if isinstance(item, dict):
                field_records[section].append(item)
            else:
                field_structure_errors.append([str(field_path), section, "dict가 아닌 항목"])

if field_read_errors or field_structure_errors:
    raise ValueError(
        f"전체 필드 집계를 완료하지 못했습니다. 읽기 오류: {field_read_errors[:3]}, "
        f"구조 오류: {field_structure_errors[:3]}"
    )

def field_type_summary(values):
    types = {type(value).__name__ for value in values}
    if "list" in types:
        elements = [
            element for value in values if isinstance(value, list)
            for element in value
        ]
        element_types = {type(element).__name__ for element in elements}
        if element_types:
            types.remove("list")
            types.add("list[" + ", ".join(sorted(element_types)) + "]")
    return ", ".join(sorted(types)) if types else "미존재"

def field_value_summary(section, key, records):
    values = [record[key] for record in records if key in record]
    null_count = sum(value is None for value in values)
    empty_count = sum(isinstance(value, str) and value == "" for value in values)
    missing_count = len(records) - len(values)
    filled = [
        value for value in values
        if value is not None and not (isinstance(value, str) and value == "")
    ]
    parts = []
    if filled and all(
        isinstance(value, (int, float)) and not isinstance(value, bool)
        for value in filled
    ):
        unique = sorted(set(filled))
        if len(unique) == 1:
            parts.append(f"{unique[0]}로 동일")
        elif len(unique) <= 10:
            parts.append(", ".join(str(value) for value in unique))
            parts.append(f"고유 값 {len(unique)}개")
        else:
            parts.append(f"관찰 범위: {min(unique)}~{max(unique)}")
            parts.append(f"고유 값 {len(unique)}개")
        if key in {"id", "image_id", "category_id", "item_seq"}:
            parts.append("식별자 번호 기준")
    elif filled and all(isinstance(value, list) for value in filled):
        lengths = sorted({len(value) for value in filled})
        tokens = list(dict.fromkeys(
            json.dumps(value, ensure_ascii=False) for value in filled
        ))
        parts.append(f"{tokens[0]}로 동일" if len(tokens) == 1 else f"예: {tokens[0]}")
        parts.append("길이 " + ", ".join(map(str, lengths)))
        # 숫자 목록은 위치별 범위만 기록한다. 좌표 의미를 추정해서 이름 붙이지 않는다.
        if (
            len(lengths) == 1 and 0 < lengths[0] <= 8
            and all(
                isinstance(element, (int, float)) and not isinstance(element, bool)
                for value in filled for element in value
            )
        ):
            ranges = []
            for index in range(lengths[0]):
                column = [value[index] for value in filled]
                ranges.append(f"{min(column)}~{max(column)}")
            parts.append("원소별 관찰 범위: [" + ", ".join(ranges) + "]")
    elif filled:
        tokens = list(dict.fromkeys(
            json.dumps(value, ensure_ascii=False, sort_keys=True) for value in filled
        ))
        if len(tokens) == 1:
            parts.append(f"{tokens[0]}로 동일")
        elif len(tokens) <= 10:
            parts.append(", ".join(tokens))
        else:
            parts.append(f"예: {tokens[0]}")
        if len(tokens) > 1:
            parts.append(f"고유 값 {len(tokens)}개")
    if empty_count:
        parts.append(f'빈 문자열 "" {empty_count}건')
    if null_count:
        parts.append(f"null {null_count}건")
    if missing_count:
        parts.append(f"key 누락 {missing_count}건")
    return "; ".join(parts) if parts else "관찰 값 없음"

field_tables = {}
field_keyset_counts = {}
for section, definitions in field_definitions.items():
    records = field_records[section]
    observed_keys = {key for record in records for key in record}
    # 정의표의 항목 외에 실제 데이터에서 발견된 key도 생략하지 않는다.
    ordered_keys = (
        [key for key in definitions if key in observed_keys]
        + sorted(observed_keys - set(definitions))
    )
    field_tables[section] = [
        [
            key,
            definitions.get(key, "—"),
            field_value_summary(section, key, records),
            field_type_summary([record[key] for record in records if key in record]),
        ]
        for key in ordered_keys
    ]
    field_keyset_counts[section] = len({
        tuple(sorted(record)) for record in records
    })


# 각 필드의 정의와 실제 값은 한 용도에만 배치한다.
field_groups = {
    "image": {
        "images": ["id", "file_name", "imgfile", "width", "height", "size"],
        "annotations": ["image_id"],
    },
    "drug": {
        "images": [
            "drug_N", "dl_idx", "dl_mapping_code", "dl_name", "dl_name_en",
            "img_key", "dl_material", "dl_material_en", "dl_company", "dl_company_en",
            "di_company_mf", "di_company_mf_en", "item_seq", "di_item_permit_date",
            "di_class_no", "di_etc_otc_code", "di_edi_code", "img_regist_ts", "change_date",
        ],
        "categories": ["id", "name", "supercategory"],
        "annotations": ["category_id"],
    },
    "appearance": {
        "images": [
            "chart", "dl_custom_shape", "drug_shape", "form_code_name",
            "thick", "leng_long", "leng_short", "print_front", "print_back",
            "color_class1", "color_class2", "line_front", "line_back",
            "mark_code_front_anal", "mark_code_back_anal",
            "mark_code_front_img", "mark_code_back_img",
            "mark_code_front", "mark_code_back",
        ],
    },
    "capture": {
        "images": ["drug_S", "back_color", "light_color", "drug_dir", "camera_la", "camera_lo"],
    },
    "object": {
        "annotations": ["id", "bbox", "area", "iscrowd", "ignore", "segmentation"],
    },
}
field_rows_by_key = {
    f"{section}.{row[0]}": row
    for section, rows in field_tables.items() for row in rows
}
grouped_keys = [
    f"{section}.{key}"
    for blocks in field_groups.values()
    for section, keys in blocks.items() for key in keys
]
if len(grouped_keys) != len(set(grouped_keys)) or set(grouped_keys) != set(field_rows_by_key):
    raise ValueError("필드 분류에 누락 또는 중복이 있습니다. 실제 key와 용도별 분류를 확인하세요.")


def show_field_group(group):
    rows = []
    for section, keys in field_groups[group].items():
        for key in keys:
            qualified_key = f"{section}.{key}"
            rows.append([qualified_key, *field_rows_by_key[qualified_key][1:]])
    show_table(["key", "정의", "실제 값·관찰 범위", "자료형"], rows)


show_table(
    ["구조", "딕셔너리 항목 수", "전체 key 수", "관찰된 key 구성 종류 수"],
    [
        [section, len(field_records[section]), len(field_tables[section]), field_keyset_counts[section]]
        for section in field_definitions
    ],
)

| 구조 | 딕셔너리 항목 수 | 전체 key 수 | 관찰된 key 구성 종류 수 |
| --- | --- | --- | --- |
| images | 763 | 50 | 1 |
| annotations | 763 | 8 | 1 |
| categories | 763 | 3 | 1 |

#### 2.1.5 분석 결과와 필요한 조치

- **관찰한 사실:** JSON 763개를 모두 읽었으며 각 파일의 images·annotations·categories에는 항목이 각각 1개 있다. 구조별 key는 50개·8개·3개이며, 전체 파일에서 key 구성이 각각 동일하다.
- **학습에 미치는 영향:** 원시 annotation은 763개지만 반복되는 이미지 정보와 클래스 정의가 있으므로 항목 합계를 고유 이미지·클래스 수로 해석할 수 없다. key 구성이 같아도 값의 결측이나 자료형은 다를 수 있다.
- **필요한 조치:** 이미지 파일명·클래스 ID로 고유 대상을 구분하고, 원시 annotation은 중복·좌표·실제 객체 대응을 검증한 후 유효 객체 수를 확정한다.

### 2.2 이미지 정보와 라벨 연결

**목적:** 이미지 식별자·파일명·크기 필드의 의미와 실제 값을 이해하고,
JSON이 가리키는 파일과 실제 학습 이미지의 연결 및 이미지별 정답 수집 기준을 확인한다.

#### 2.2.1 이미지 필드의 정의·실제 값·자료형


In [77]:
show_field_group("image")

| key | 정의 | 실제 값·관찰 범위 | 자료형 |
| --- | --- | --- | --- |
| images.id | 약제 이미지식별자 | 관찰 범위: 14~1499; 고유 값 232개; 식별자 번호 기준 | int |
| images.file_name | 약제 이미지파일명 | 예: "K-001900-016548-019607-029451_0_2_0_2_70_000_200.png"; 고유 값 232개 | str |
| images.imgfile | — | 예: "K-001900-016548-019607-029451_0_2_0_2_70_000_200.png"; 고유 값 232개 | str |
| images.width | 약제 이미지너비 | 976로 동일 | int |
| images.height | 약제 이미지높이 | 1280로 동일 | int |
| images.size | 이미지 사이즈 | 200로 동일 | int |
| annotations.image_id | 이미지 아이디 | 관찰 범위: 14~1499; 고유 값 232개; 식별자 번호 기준 | int |

`width`·`height`는 이미지의 가로·세로 크기를 나타내는 별도 필드이다.

`size`의 공식 명칭은 이미지 사이즈지만, 값 200의 단위·산정 방식은 확인되지 않았다.

따라서 `size`를 이미지 가로·세로 픽셀 수나 리사이즈 목표로 해석할 근거는 아직 없다.

`annotations.image_id`는 같은 JSON의 `images.id`를 참조한다. 이미지 ID와 파일명의 대응은 전체 JSON을 기준으로 확인한다.

#### 2.2.2 이미지 크기 필드의 제공 범위와 일관성

**목적:** 같은 이미지가 여러 JSON에 기록될 때 크기 정보가 빠지거나 달라지는지 확인한다.

In [64]:
metadata_fields = {
    "width": "기록된 이미지 너비",
    "height": "기록된 이미지 높이",
    "back_color": "배경색",
    "light_color": "조명",
    "drug_dir": "알약 앞뒷면",
    "camera_la": "카메라 관련 필드",
    "camera_lo": "카메라 관련 필드",
    "size": "size 필드(의미·단위 추가 확인)",
}
metadata_summary = []
metadata_details = {}

for field, description in metadata_fields.items():
    filled_count = 0
    value_tokens = set()
    values_by_image = defaultdict(set)
    for row in image_rows:
        value = row["metadata"].get(field)
        if value is None or (isinstance(value, str) and not value.strip()):
            continue
        filled_count += 1
        # 값의 자료형을 유지하며 비교한다.
        token = json.dumps(value, ensure_ascii=False, sort_keys=True)
        value_tokens.add(token)
        if row["image_name"]:
            values_by_image[row["image_name"]].add(token)

    value_examples = [str(json.loads(token)) for token in sorted(value_tokens)]
    conflict_count = sum(len(values) > 1 for values in values_by_image.values())
    metadata_details[field] = {
        "filled_records": filled_count,
        "unique_values": len(value_tokens),
        "images_with_value": len(values_by_image),
        "images_with_conflicts": conflict_count,
    }
    metadata_summary.append([
        field,
        description,
        f"{filled_count}/{len(image_rows)}",
        len(image_rows) - filled_count,
        len(values_by_image),
        len(value_tokens),
        ", ".join(value_examples[:5]),
        conflict_count,
    ])


def show_metadata_consistency(fields):
    show_table(
        ["key", "값 있는 images 항목", "결측 항목", "값 있는 고유 이미지", "동일 이미지 내 값 불일치 수"],
        [
            [f"images.{row[0]}", row[2], row[3], row[4], row[7]]
            for row in metadata_summary if row[0] in fields
        ],
    )


show_metadata_consistency({"width", "height", "size"})

| key | 값 있는 images 항목 | 결측 항목 | 값 있는 고유 이미지 | 동일 이미지 내 값 불일치 수 |
| --- | --- | --- | --- | --- |
| images.width | 763/763 | 0 | 232 | 0 |
| images.height | 763/763 | 0 | 232 | 0 |
| images.size | 763/763 | 0 | 232 | 0 |

#### 2.2.3 실제 이미지 파일과 JSON의 참조 관계

**목적:** 파일명 기준 참조 누락과 중복 파일명을 확인하고 이미지 한 장에 연결된 JSON 수를 파악한다.

In [65]:
train_files_by_name = defaultdict(list)
test_files_by_name = defaultdict(list)
for image_path in train_image_paths:
    train_files_by_name[image_path.name].append(image_path)
for image_path in test_image_paths:
    test_files_by_name[image_path.name].append(image_path)

jsons_by_image = defaultdict(set)
missing_file_name_rows = []
for row in image_rows:
    if row["image_name"] is None:
        missing_file_name_rows.append(row["json_path"])
    else:
        jsons_by_image[row["image_name"]].add(row["json_path"])

train_names = set(train_files_by_name)
referenced_names = set(jsons_by_image)
ambiguous_train_names = {
    name for name, paths in train_files_by_name.items() if len(paths) > 1
}
ambiguous_test_names = {
    name for name, paths in test_files_by_name.items() if len(paths) > 1
}
train_without_reference = train_names - referenced_names
references_without_train = referenced_names - train_names

show_table(
    ["확인 항목", "개수"],
    [
        ["실제 학습 이미지 파일", len(train_image_paths)],
        ["JSON이 가리키는 고유 이미지 파일명", len(referenced_names)],
        ["학습 이미지에서 중복된 파일명", len(ambiguous_train_names)],
        ["테스트 이미지에서 중복된 파일명", len(ambiguous_test_names)],
        ["file_name이 없거나 빈 images 항목", len(missing_file_name_rows)],
        ["JSON 참조가 없는 학습 이미지 파일명", len(train_without_reference)],
        ["학습 이미지 폴더에서 찾지 못한 참조 파일명", len(references_without_train)],
    ],
)

# 파일명이 고유할 때만 '이미지당 JSON 수'로 해석한다.
if ambiguous_train_names:
    display(Markdown("**주의:** 파일명 중복이 있어 경로를 포함한 연결 기준이 필요합니다."))
else:
    json_count_distribution = Counter(
        len(jsons_by_image.get(name, set())) for name in train_names
    )
    show_table(
        ["학습 이미지 한 장에 연결된 JSON 수", "이미지 수"],
        sorted(json_count_distribution.items()),
    )

if train_without_reference or references_without_train:
    show_table(
        ["종류", "파일명 예시(최대 5개)"],
        [
            ["JSON 참조가 없는 이미지", ", ".join(sorted(train_without_reference)[:5])],
            ["이미지가 없는 JSON 참조", ", ".join(sorted(references_without_train)[:5])],
        ],
    )

| 확인 항목 | 개수 |
| --- | --- |
| 실제 학습 이미지 파일 | 232 |
| JSON이 가리키는 고유 이미지 파일명 | 232 |
| 학습 이미지에서 중복된 파일명 | 0 |
| 테스트 이미지에서 중복된 파일명 | 0 |
| file_name이 없거나 빈 images 항목 | 0 |
| JSON 참조가 없는 학습 이미지 파일명 | 0 |
| 학습 이미지 폴더에서 찾지 못한 참조 파일명 | 0 |

| 학습 이미지 한 장에 연결된 JSON 수 | 이미지 수 |
| --- | --- |
| 2 | 7 |
| 3 | 151 |
| 4 | 74 |

#### 2.2.4 JSON 내부의 이미지 ID 연결과 전체 ID 범위

**목적:** 각 객체 정답이 같은 JSON의 이미지 정보를 가리키는지 확인하고,
이미지 ID와 파일명의 대응을 근거로 여러 JSON을 함께 읽을 기준을 정한다.

- 내부 연결: `annotations.image_id`와 **같은 JSON**의 `images.id`를 비교한다.
- 전체 대응: 하나의 이미지 ID가 여러 파일명에 쓰이는지, 하나의 파일명에 여러 ID가 기록되는지 확인한다.
- 같은 이미지가 여러 JSON에 반복 기록되는 것은 서로 다른 이미지에 ID가 재사용되는 경우와 구분한다.

**집계 단위:** 객체 연결은 annotation 항목 수, ID 재사용은 고유 ID 수,
파일명의 ID 불일치는 고유 파일명 수로 센다. ID는 정수 여부를 확인하며 원본 값을 변환하지 않는다.

In [66]:
def valid_connection_id(value):
    return isinstance(value, int) and not isinstance(value, bool)


# JSON 경로를 포함해 조회하므로 다른 파일의 같은 번호와 연결되지 않는다.
images_by_json_id = defaultdict(list)
image_files_by_id = defaultdict(set)
image_ids_by_file = defaultdict(set)
invalid_image_id_rows = []
for row in image_rows:
    image_id = row["metadata"].get("id")
    if not valid_connection_id(image_id):
        invalid_image_id_rows.append(row)
        continue
    images_by_json_id[(row["json_path"], image_id)].append(row)
    if row["image_name"]:
        image_files_by_id[image_id].add(row["image_name"])
        image_ids_by_file[row["image_name"]].add(image_id)

image_link_checks = []
for annotation in annotation_rows:
    image_id = annotation.get("image_id")
    matches = (
        images_by_json_id.get((annotation["json_path"], image_id), [])
        if valid_connection_id(image_id) else []
    )
    if not valid_connection_id(image_id):
        status = "image_id 누락·형식 확인"
    elif not matches:
        status = "같은 JSON에서 이미지 ID를 찾지 못함"
    elif len(matches) > 1:
        status = "같은 JSON에서 이미지 ID가 여러 항목과 연결됨"
    else:
        status = "일치"
    image_link_checks.append({
        "json_path": annotation["json_path"],
        "annotation": annotation,
        "image_id": image_id,
        "image": matches[0] if status == "일치" else None,
        "status": status,
    })

image_link_status_counts = Counter(row["status"] for row in image_link_checks)
image_id_file_conflicts = {
    image_id: files for image_id, files in image_files_by_id.items() if len(files) > 1
}
image_file_id_conflicts = {
    name: ids for name, ids in image_ids_by_file.items() if len(ids) > 1
}
show_table(
    ["검사 항목", "결과"],
    [
        ["검사한 객체 정답 수", len(image_link_checks)],
        ["같은 JSON의 이미지와 연결된 객체 정답 수", image_link_status_counts["일치"]],
        ["id가 없거나 정수가 아닌 images 항목 수", len(invalid_image_id_rows)],
        ["image_id가 없거나 정수가 아닌 객체 정답 수", image_link_status_counts["image_id 누락·형식 확인"]],
        ["같은 JSON에서 이미지 ID를 찾지 못한 객체 정답 수", image_link_status_counts["같은 JSON에서 이미지 ID를 찾지 못함"]],
        ["같은 JSON에서 이미지 ID가 여러 항목과 연결된 객체 정답 수", image_link_status_counts["같은 JSON에서 이미지 ID가 여러 항목과 연결됨"]],
        ["고유 이미지 ID 수", len(image_files_by_id)],
        ["고유 이미지 파일명 수", len(image_ids_by_file)],
        ["서로 다른 파일명에 재사용된 이미지 ID 수", len(image_id_file_conflicts)],
        ["여러 이미지 ID가 기록된 파일명 수", len(image_file_id_conflicts)],
    ],
)

# 확인 대상은 종류별 최대 5건을 보여주고, 위 요약표에 전체 건수를 남긴다.
for status, count in sorted(image_link_status_counts.items()):
    if status == "일치":
        continue
    display(Markdown(f"**{status}: 전체 {count}건, 최대 5건 표시**"))
    show_table(
        ["JSON 경로(data/raw 기준)", "annotation ID", "image_id"],
        [
            [row["json_path"], row["annotation"].get("id"), row["image_id"]]
            for row in image_link_checks if row["status"] == status
        ][:5],
    )
if invalid_image_id_rows:
    show_table(
        ["id 형식 확인이 필요한 JSON(최대 5건)", "images.id"],
        [[row["json_path"], row["metadata"].get("id")] for row in invalid_image_id_rows[:5]],
    )
if image_id_file_conflicts:
    show_table(
        ["재사용된 이미지 ID(최대 5개)", "연결된 서로 다른 파일명"],
        [[key, " / ".join(sorted(values))] for key, values in sorted(image_id_file_conflicts.items())[:5]],
    )
if image_file_id_conflicts:
    show_table(
        ["여러 ID가 기록된 파일명(최대 5개)", "이미지 ID 목록"],
        [[key, sorted(values)] for key, values in sorted(image_file_id_conflicts.items())[:5]],
    )

| 검사 항목 | 결과 |
| --- | --- |
| 검사한 객체 정답 수 | 763 |
| 같은 JSON의 이미지와 연결된 객체 정답 수 | 763 |
| id가 없거나 정수가 아닌 images 항목 수 | 0 |
| image_id가 없거나 정수가 아닌 객체 정답 수 | 0 |
| 같은 JSON에서 이미지 ID를 찾지 못한 객체 정답 수 | 0 |
| 같은 JSON에서 이미지 ID가 여러 항목과 연결된 객체 정답 수 | 0 |
| 고유 이미지 ID 수 | 232 |
| 고유 이미지 파일명 수 | 232 |
| 서로 다른 파일명에 재사용된 이미지 ID 수 | 0 |
| 여러 이미지 ID가 기록된 파일명 수 | 0 |

#### 2.2.5 이미지별 실제 연결 예시

**목적:** 한 이미지의 정보가 여러 JSON에 반복되고 각 객체 정답이 해당 이미지 ID로 연결되는 모습을 확인한다.

연결된 JSON 수가 2개·3개·4개인 이미지에서 각각 한 장을 선택한다.
각 그룹에서 파일명 정렬상 첫 이미지를 사용하므로 실행할 때마다 같은 예시를 볼 수 있다.
아래 표에는 선택한 이미지에 연결된 모든 annotation을 표시하며, 전체 연결 검사는 앞의 요약표 기준이다.

In [67]:
image_link_samples = []
for target_json_count in [2, 3, 4]:
    candidates = sorted(
        name for name, paths in jsons_by_image.items()
        if len(paths) == target_json_count and name in train_names
    )
    if not candidates:
        display(Markdown(f"연결 JSON이 {target_json_count}개인 학습 이미지가 없습니다."))
        continue
    selected_name = candidates[0]
    selected_paths = jsons_by_image[selected_name]
    selected_links = [
        row for row in image_link_checks if row["json_path"] in selected_paths
    ]
    image_link_samples.append({
        "image_name": selected_name,
        "json_count": target_json_count,
        "annotation_count": len(selected_links),
    })
    display(Markdown(
        f"**연결 JSON {target_json_count}개인 이미지**\n\n`{selected_name}`"
    ))
    show_table(
        ["JSON 경로(data/raw 기준)", "images.id", "annotations.image_id", "같은 JSON 내 연결"],
        [
            [
                row["json_path"],
                row["image"]["metadata"].get("id") if row["image"] else "연결 미확정",
                row["image_id"], row["status"],
            ]
            for row in sorted(selected_links, key=lambda row: row["json_path"])
        ],
    )

**연결 JSON 2개인 이미지**

`K-003351-013900-021325_0_2_0_2_70_000_200.png`

| JSON 경로(data/raw 기준) | images.id | annotations.image_id | 같은 JSON 내 연결 |
| --- | --- | --- | --- |
| train_annotations/K-003351-013900-021325_json/K-013900/K-003351-013900-021325_0_2_0_2_70_000_200.json | 193 | 193 | 일치 |
| train_annotations/K-003351-013900-021325_json/K-021325/K-003351-013900-021325_0_2_0_2_70_000_200.json | 193 | 193 | 일치 |

**연결 JSON 3개인 이미지**

`K-003351-003832-016232_0_2_0_2_75_000_200.png`

| JSON 경로(data/raw 기준) | images.id | annotations.image_id | 같은 JSON 내 연결 |
| --- | --- | --- | --- |
| train_annotations/K-003351-003832-016232_json/K-003351/K-003351-003832-016232_0_2_0_2_75_000_200.json | 281 | 281 | 일치 |
| train_annotations/K-003351-003832-016232_json/K-003832/K-003351-003832-016232_0_2_0_2_75_000_200.json | 281 | 281 | 일치 |
| train_annotations/K-003351-003832-016232_json/K-016232/K-003351-003832-016232_0_2_0_2_75_000_200.json | 281 | 281 | 일치 |

**연결 JSON 4개인 이미지**

`K-001900-016548-019607-029451_0_2_0_2_70_000_200.png`

| JSON 경로(data/raw 기준) | images.id | annotations.image_id | 같은 JSON 내 연결 |
| --- | --- | --- | --- |
| train_annotations/K-001900-016548-019607-029451_json/K-001900/K-001900-016548-019607-029451_0_2_0_2_70_000_200.json | 34 | 34 | 일치 |
| train_annotations/K-001900-016548-019607-029451_json/K-016548/K-001900-016548-019607-029451_0_2_0_2_70_000_200.json | 34 | 34 | 일치 |
| train_annotations/K-001900-016548-019607-029451_json/K-019607/K-001900-016548-019607-029451_0_2_0_2_70_000_200.json | 34 | 34 | 일치 |
| train_annotations/K-001900-016548-019607-029451_json/K-029451/K-001900-016548-019607-029451_0_2_0_2_70_000_200.json | 34 | 34 | 일치 |

#### 2.2.6 파일명 코드와 JSON 이미지 코드 비교

**목적:** 파일명에서 읽은 약품 코드 후보와 연결된 JSON의 `images.drug_N`을 비교해, JSON에 나타나지 않는 코드가 있는 이미지 수와 코드 수를 집계한다.

파일명에 적힌 코드 의미가 공식적으로 모두 확인된 것은 아니므로, 차이는 **라벨 누락 확정이 아닌 누락 의심 후보**로 해석한다. 실제 이미지에서 해당 약품이 보이는지 확인한 뒤 누락 여부를 판단한다.


In [82]:
import re

# 파일명 앞부분의 K-숫자 코드 목록과 연결 JSON의 images.drug_N을 비교한다.
drug_codes_by_image = defaultdict(set)
for row in image_rows:
    image_name = row.get("image_name")
    drug_code = row["metadata"].get("drug_N")
    if image_name and isinstance(drug_code, str) and drug_code.strip():
        drug_codes_by_image[image_name].add(drug_code.strip())

filename_json_code_checks = []
unparsed_filename_rows = []
for image_path in train_image_paths:
    image_name = image_path.name
    filename_prefix = image_path.stem.split("_", 1)[0]
    if not re.fullmatch(r"K(?:-\d{6})+", filename_prefix):
        unparsed_filename_rows.append(image_name)
        continue

    filename_codes = {
        f"K-{code}" for code in filename_prefix.split("-")[1:]
    }
    json_codes = drug_codes_by_image.get(image_name, set())
    missing_codes = filename_codes - json_codes
    extra_codes = json_codes - filename_codes
    filename_json_code_checks.append({
        "image_name": image_name,
        "filename_codes": filename_codes,
        "json_codes": json_codes,
        "missing_codes": missing_codes,
        "extra_codes": extra_codes,
        "json_count": len(jsons_by_image.get(image_name, set())),
    })

suspected_missing_rows = [
    row for row in filename_json_code_checks if row["missing_codes"]
]
extra_json_code_rows = [
    row for row in filename_json_code_checks if row["extra_codes"]
]
filename_json_code_count_distribution = Counter(
    (len(row["filename_codes"]), len(row["json_codes"]))
    for row in filename_json_code_checks
)

show_table(
    ["확인 항목", "결과"],
    [
        ["파일명 코드 추출에 성공한 학습 이미지", len(filename_json_code_checks)],
        ["파일명 코드가 JSON에도 모두 나타나는 이미지", sum(not row["missing_codes"] for row in filename_json_code_checks)],
        ["누락 의심 코드가 하나 이상 있는 이미지", len(suspected_missing_rows)],
        ["누락 의심 코드 수", sum(len(row["missing_codes"]) for row in suspected_missing_rows)],
        ["파일명에는 없고 JSON에만 있는 코드 수", sum(len(row["extra_codes"]) for row in extra_json_code_rows)],
        ["파일명 규칙을 읽지 못한 이미지", len(unparsed_filename_rows)],
    ],
)

show_table(
    ["파일명 코드 수", "JSON drug_N 코드 수", "이미지 수"],
    [
        [filename_count, json_count, count]
        for (filename_count, json_count), count
        in sorted(filename_json_code_count_distribution.items())
    ],
)

if suspected_missing_rows:
    show_table(
        ["이미지 파일명", "파일명 코드 후보", "JSON drug_N 코드", "누락 의심 코드", "연결 JSON 수"],
        [
            [
                row["image_name"],
                ", ".join(sorted(row["filename_codes"])),
                ", ".join(sorted(row["json_codes"])),
                ", ".join(sorted(row["missing_codes"])),
                row["json_count"],
            ]
            for row in sorted(suspected_missing_rows, key=lambda item: item["image_name"])
        ],
    )

if extra_json_code_rows:
    show_table(
        ["이미지 파일명", "파일명 코드 후보", "JSON에만 있는 코드"],
        [
            [row["image_name"], ", ".join(sorted(row["filename_codes"])), ", ".join(sorted(row["extra_codes"]))]
            for row in sorted(extra_json_code_rows, key=lambda item: item["image_name"])
        ],
    )

if unparsed_filename_rows:
    show_table(
        ["파일명 규칙 확인이 필요한 이미지"],
        [[name] for name in unparsed_filename_rows[:10]],
    )


| 확인 항목 | 결과 |
| --- | --- |
| 파일명 코드 추출에 성공한 학습 이미지 | 232 |
| 파일명 코드가 JSON에도 모두 나타나는 이미지 | 224 |
| 누락 의심 코드가 하나 이상 있는 이미지 | 8 |
| 누락 의심 코드 수 | 8 |
| 파일명에는 없고 JSON에만 있는 코드 수 | 0 |
| 파일명 규칙을 읽지 못한 이미지 | 0 |

| 파일명 코드 수 | JSON drug_N 코드 수 | 이미지 수 |
| --- | --- | --- |
| 3 | 2 | 7 |
| 3 | 3 | 150 |
| 4 | 3 | 1 |
| 4 | 4 | 74 |

| 이미지 파일명 | 파일명 코드 후보 | JSON drug_N 코드 | 누락 의심 코드 | 연결 JSON 수 |
| --- | --- | --- | --- | --- |
| K-003351-013900-021325_0_2_0_2_70_000_200.png | K-003351, K-013900, K-021325 | K-013900, K-021325 | K-003351 | 2 |
| K-003351-013900-036637_0_2_0_2_70_000_200.png | K-003351, K-013900, K-036637 | K-013900, K-036637 | K-003351 | 2 |
| K-003351-020014-022074_0_2_0_2_90_000_200.png | K-003351, K-020014, K-022074 | K-003351, K-022074 | K-020014 | 2 |
| K-003351-021325-032310_0_2_0_2_90_000_200.png | K-003351, K-021325, K-032310 | K-003351, K-021325 | K-032310 | 2 |
| K-003351-032310-038162_0_2_0_2_70_000_200.png | K-003351, K-032310, K-038162 | K-032310, K-038162 | K-003351 | 2 |
| K-003351-033880-038162_0_2_0_2_75_000_200.png | K-003351, K-033880, K-038162 | K-003351, K-038162 | K-033880 | 2 |
| K-003351-035206-041768_0_2_0_2_70_000_200.png | K-003351, K-035206, K-041768 | K-035206, K-041768 | K-003351 | 2 |
| K-003544-004543-012247-016548_0_2_0_2_90_000_200.png | K-003544, K-004543, K-012247, K-016548 | K-003544, K-012247, K-016548 | K-004543 | 3 |

#### 2.2.7 분석 결과와 필요한 조치

- **관찰한 사실:** JSON 763개는 고유 학습 이미지 232개를 가리키며 파일명 기준 참조 누락과 중복 파일명은 없다. 이미지당 JSON은 2~4개다. 기록된 크기는 width=976·height=1280·size=200이며, 세 필드는 결측과 동일 이미지 내 값 불일치가 없다. 객체 정답 763개 모두 같은 JSON의 이미지 ID와 연결되고, 고유 이미지 ID 232개와 파일명 232개는 일대일로 대응한다.
- **학습에 미치는 영향:** JSON 하나만 읽으면 같은 이미지의 다른 객체 정답을 빠뜨릴 수 있다. 현재 데이터에서는 이미지 ID가 서로 다른 파일명에 재사용되는 경우가 없고, 같은 이미지의 여러 JSON이 같은 ID를 공유한다. JSON 단위 분할은 동일 이미지를 학습·검증 양쪽에 포함시킬 수 있다. 기록된 크기가 같다는 결과만으로 모든 실제 이미지 크기가 검증된 것은 아니다.
- **필요한 조치:** 현재 데이터에서 고유성이 확인된 이미지 파일명을 기준으로 관련 JSON의 객체 정답을 함께 조회하고 원본 ID 대응을 보관한다. 같은 이미지의 모든 정답은 같은 분할에 포함한다. 실제 이미지 크기는 이미지 파일을 읽어 대조하며 size는 의미가 확인될 때까지 크기 변환 기준으로 사용하지 않는다.

**후속 확인:** **3번**에서 실제 객체와 박스를 대조하고 **5번**에서 박스·면적·중복을 검사한다. **6번**에서 동일·유사 촬영 이미지의 분할 기준을 정한다.

### 2.3 약품·클래스 정보

**목적:** 약품 코드·제품명·성분·제조사 등 기본 정보와 클래스 정의를 이해하고,
객체 정답의 클래스 참조와 학습용 ID 관리에 필요한 기준을 확인한다.

#### 2.3.1 약품·클래스 필드의 정의·실제 값·자료형


In [78]:
show_field_group("drug")

| key | 정의 | 실제 값·관찰 범위 | 자료형 |
| --- | --- | --- | --- |
| images.drug_N | 알약코드 | 예: "K-001900"; 고유 값 56개 | str |
| images.dl_idx | 알약 식별자 | 예: "1900"; 고유 값 56개 | str |
| images.dl_mapping_code | 제품코드 | 예: "K-001900"; 고유 값 56개 | str |
| images.dl_name | 제품명 | 예: "보령부스파정 5mg"; 고유 값 56개 | str |
| images.dl_name_en | 제품명(영어) | 예: "Buspar Tab. 5mg Boryung"; 고유 값 55개; 빈 문자열 "" 14건 | str |
| images.img_key | 이미지 링크 | 예: "http://connectdi.com/design/img/drug/1Mxwka5v0lL.jpg"; 고유 값 55개; null 18건 | NoneType, str |
| images.dl_material | 성분명 | 예: "부스피론염산염"; 고유 값 49개; 빈 문자열 "" 9건 | str |
| images.dl_material_en | 성분명(영어) | 예: "Buspirone Hydrochloride"; 고유 값 51개 | str |
| images.dl_company | 제조사명 | 예: "보령제약(주)"; 고유 값 30개 | str |
| images.dl_company_en | 제조사명(영어) | 예: "Boryung"; 고유 값 30개 | str |
| images.di_company_mf | 위탁제조사명 | 예: "일양바이오팜(주)"; 고유 값 13개; 빈 문자열 "" 637건 | str |
| images.di_company_mf_en | 위탁제조사명(영어) | "Ilyang Bio Pharm", "Korean Drug", "Kolon Pharm", "Hlb Pharma", "Suheung", "Suheung, Rp Bio, Cosmax Pharma"; 고유 값 6개; 빈 문자열 "" 724건 | str |
| images.item_seq | 품목기준코드 | 관찰 범위: 198700706~201702674; 고유 값 56개; 식별자 번호 기준 | int |
| images.di_item_permit_date | 허가일자 | 예: "19870323"; 고유 값 56개 | str |
| images.di_class_no | 약품 분류 | 예: "[01170]정신신경용제"; 고유 값 13개 | str |
| images.di_etc_otc_code | 전문의약품/ 일반의약품 | "전문의약품", "일반의약품"; 고유 값 2개 | str |
| images.di_edi_code | EDI 코드 | 예: "641901280,A09302381"; 고유 값 56개 | str |
| images.img_regist_ts | 약학정보원 이미지 생성일 | 예: "20070910"; 고유 값 44개; null 18건 | NoneType, str |
| images.change_date | 변경일자 | 예: "20160825"; 고유 값 45개; null 18건 | NoneType, str |
| categories.id | 카테고리 아이디 | 관찰 범위: 1900~41768; 고유 값 56개; 식별자 번호 기준 | int |
| categories.name | 카테고리 명 | 예: "보령부스파정 5mg"; 고유 값 56개 | str |
| categories.supercategory | 슈퍼 카테고리 | "pill"로 동일 | str |
| annotations.category_id | category 아이디 | 관찰 범위: 1900~41768; 고유 값 56개; 식별자 번호 기준 | int |

#### 2.3.2 클래스 목록과 원본 ID의 일관성

**목적:** 클래스 수·ID와 이름의 대응·annotation의 클래스 참조를 확인하고 학습용 인덱스의 필요성을 판단한다.

표의 **순번(확인용)**은 약품 목록을 세기 위한 번호이며, 원본 클래스 ID와 구분한다.

In [69]:
names_by_class_id = defaultdict(set)
invalid_category_rows = []
for row in category_rows:
    class_id = row.get("id")
    class_name = row.get("name")
    if (
        not isinstance(class_id, int) or isinstance(class_id, bool)
        or not isinstance(class_name, str) or not class_name.strip()
    ):
        invalid_category_rows.append(row)
        continue
    # 원본 ID와 이름을 그대로 보존한다.
    names_by_class_id[class_id].add(class_name)

annotation_class_ids = {
    row["category_id"] for row in annotation_rows
    if isinstance(row.get("category_id"), int)
    and not isinstance(row.get("category_id"), bool)
}
invalid_annotation_category_count = sum(
    not isinstance(row.get("category_id"), int)
    or isinstance(row.get("category_id"), bool)
    for row in annotation_rows
)
original_class_ids = sorted(names_by_class_id)
class_name_conflicts = {
    class_id: names
    for class_id, names in names_by_class_id.items() if len(names) > 1
}
undefined_class_ids = annotation_class_ids - set(original_class_ids)
is_zero_based_contiguous = original_class_ids == list(range(len(original_class_ids)))
class_catalog = [
    [class_id, " / ".join(sorted(names_by_class_id[class_id]))]
    for class_id in original_class_ids
]

show_table(
    ["확인 항목", "결과"],
    [
        ["categories의 고유 클래스 ID 수", len(original_class_ids)],
        ["annotations에 등장하는 고유 클래스 ID 수", len(annotation_class_ids)],
        ["클래스 ID 범위", f"{min(original_class_ids)} ~ {max(original_class_ids)}" if original_class_ids else "없음"],
        ["0부터 시작하는 연속 ID인가?", is_zero_based_contiguous],
        ["같은 ID에 이름이 여러 개인 경우", len(class_name_conflicts)],
        ["categories에 정의되지 않은 annotation 클래스 ID", len(undefined_class_ids)],
        ["유효한 ID·이름을 확인하지 못한 categories 항목", len(invalid_category_rows)],
        ["유효한 category_id가 없는 annotations 항목", invalid_annotation_category_count],
    ],
)
show_table(
    ["순번(확인용)", "원본 클래스 ID", "약품명(원본 표기)"],
    [[number, *row] for number, row in enumerate(class_catalog, start=1)],
)
if class_name_conflicts:
    show_table(
        ["확인할 클래스 ID", "연결된 이름"],
        [[key, " / ".join(sorted(value))] for key, value in class_name_conflicts.items()],
    )

| 확인 항목 | 결과 |
| --- | --- |
| categories의 고유 클래스 ID 수 | 56 |
| annotations에 등장하는 고유 클래스 ID 수 | 56 |
| 클래스 ID 범위 | 1900 ~ 41768 |
| 0부터 시작하는 연속 ID인가? | False |
| 같은 ID에 이름이 여러 개인 경우 | 0 |
| categories에 정의되지 않은 annotation 클래스 ID | 0 |
| 유효한 ID·이름을 확인하지 못한 categories 항목 | 0 |
| 유효한 category_id가 없는 annotations 항목 | 0 |

| 순번(확인용) | 원본 클래스 ID | 약품명(원본 표기) |
| --- | --- | --- |
| 1 | 1900 | 보령부스파정 5mg |
| 2 | 2483 | 뮤테란캡슐 100mg |
| 3 | 3351 | 일양하이트린정 2mg |
| 4 | 3483 | 기넥신에프정(은행엽엑스)(수출용) |
| 5 | 3544 | 무코스타정(레바미피드)(비매품) |
| 6 | 3743 | 알드린정 |
| 7 | 3832 | 뉴로메드정(옥시라세탐) |
| 8 | 4543 | 에어탈정(아세클로페낙) |
| 9 | 12081 | 리렉스펜정 300mg/PTP |
| 10 | 12247 | 아빌리파이정 10mg |
| 11 | 12778 | 다보타민큐정 10mg/병 |
| 12 | 13395 | 써스펜8시간이알서방정 650mg |
| 13 | 13900 | 에빅사정(메만틴염산염)(비매품) |
| 14 | 16232 | 리피토정 20mg |
| 15 | 16262 | 크레스토정 20mg |
| 16 | 16548 | 가바토파정 100mg |
| 17 | 16551 | 동아가바펜틴정 800mg |
| 18 | 16688 | 오마코연질캡슐(오메가-3-산에틸에스테르90) |
| 19 | 18147 | 리리카캡슐 150mg |
| 20 | 18357 | 종근당글리아티린연질캡슐(콜린알포세레이트)  |
| 21 | 19232 | 콜리네이트연질캡슐 400mg |
| 22 | 19552 | 트루비타정 60mg/병 |
| 23 | 19607 | 스토가정 10mg |
| 24 | 19861 | 노바스크정 5mg |
| 25 | 20014 | 마도파정 |
| 26 | 20238 | 플라빅스정 75mg |
| 27 | 20877 | 엑스포지정 5/160mg |
| 28 | 21325 | 아토르바정 10mg |
| 29 | 21771 | 라비에트정 20mg |
| 30 | 22074 | 리피로우정 20mg |
| 31 | 22347 | 자누비아정 50mg |
| 32 | 22362 | 맥시부펜이알정 300mg |
| 33 | 24850 | 놀텍정 10mg |
| 34 | 25367 | 자누메트정 50/850mg |
| 35 | 25438 | 큐시드정 31.5mg/PTP |
| 36 | 25469 | 아모잘탄정 5/100mg |
| 37 | 27733 | 트윈스타정 40/5mg |
| 38 | 27777 | 카나브정 60mg |
| 39 | 27926 | 울트라셋이알서방정 |
| 40 | 27993 | 졸로푸트정 100mg |
| 41 | 28763 | 트라젠타정(리나글립틴) |
| 42 | 29345 | 비모보정 500/20mg |
| 43 | 29451 | 레일라정 |
| 44 | 29667 | 리바로정 4mg |
| 45 | 30308 | 트라젠타듀오정 2.5/850mg |
| 46 | 31863 | 아질렉트정(라사길린메실산염) |
| 47 | 31885 | 자누메트엑스알서방정 100/1000mg |
| 48 | 32310 | 글리아타민연질캡슐 |
| 49 | 33009 | 신바로정 |
| 50 | 33208 | 에스원엠프정 20mg |
| 51 | 33880 | 글리틴정(콜린알포세레이트) |
| 52 | 34597 | 제미메트서방정 50/1000mg |
| 53 | 35206 | 아토젯정 10/40mg |
| 54 | 36637 | 로수젯정10/5밀리그램 |
| 55 | 38162 | 로수바미브정 10/20mg |
| 56 | 41768 | 카발린캡슐 25mg |

#### 2.3.3 객체별 클래스 참조와 약품 정보의 대응

**목적:** 클래스 ID의 전체 목록뿐 아니라 각 객체가 같은 JSON의 올바른 클래스 정의를 가리키는지 확인하고,
약품 코드·약품명·클래스 ID의 대응을 파악한다.

- 각 annotation의 `category_id`를 **같은 JSON**의 `categories.id`와 연결한다.
- 이미지와 클래스가 각각 하나로 연결된 객체에서 `images.drug_N`과 클래스 ID의 대응을 양방향으로 집계한다.
- `images.dl_name`과 `categories.name`은 원문 그대로 비교한다. 이름 표기 차이를 곧바로 오라벨로 판정하지 않는다.
- 약품 코드의 숫자를 잘라 클래스 ID로 바꾸지 않고, 실제로 연결된 기록을 기준으로 비교한다.

**집계 단위:** 연결·이름 비교는 annotation 항목 수, 여러 값과 연결되는 경우는 고유 약품 코드·클래스 ID 수로 센다.

In [70]:
categories_by_json_id = defaultdict(list)
invalid_local_category_rows = []
for category in category_rows:
    category_id = category.get("id")
    if valid_connection_id(category_id):
        categories_by_json_id[(category["json_path"], category_id)].append(category)
    else:
        invalid_local_category_rows.append(category)

category_link_checks = []
drug_class_links = []
for image_check in image_link_checks:
    annotation = image_check["annotation"]
    category_id = annotation.get("category_id")
    matches = (
        categories_by_json_id.get((annotation["json_path"], category_id), [])
        if valid_connection_id(category_id) else []
    )
    if not valid_connection_id(category_id):
        status = "category_id 누락·형식 확인"
    elif not matches:
        status = "같은 JSON에서 클래스 ID를 찾지 못함"
    elif len(matches) > 1:
        status = "같은 JSON에서 클래스 ID가 여러 항목과 연결됨"
    else:
        status = "일치"
    category_link_checks.append({
        "json_path": annotation["json_path"],
        "annotation": annotation,
        "category_id": category_id,
        "status": status,
    })
    if image_check["status"] == "일치" and status == "일치":
        image = image_check["image"]["metadata"]
        drug_class_links.append({
            "json_path": annotation["json_path"],
            "drug_N": image.get("drug_N"),
            "category_id": category_id,
            "category_name": matches[0].get("name"),
            "drug_name": image.get("dl_name"),
        })

def present_connection_text(value):
    return isinstance(value, str) and bool(value.strip())


class_ids_by_drug_code = defaultdict(set)
drug_codes_by_class_id = defaultdict(set)
drug_names_by_class_id = defaultdict(set)
missing_drug_code_links = []
missing_drug_name_links = []
comparable_drug_names = []
for row in drug_class_links:
    if present_connection_text(row["drug_N"]):
        class_ids_by_drug_code[row["drug_N"]].add(row["category_id"])
        drug_codes_by_class_id[row["category_id"]].add(row["drug_N"])
    else:
        missing_drug_code_links.append(row)
    if present_connection_text(row["drug_name"]):
        drug_names_by_class_id[row["category_id"]].add(row["drug_name"])
    if present_connection_text(row["drug_name"]) and present_connection_text(row["category_name"]):
        comparable_drug_names.append(row)
    else:
        missing_drug_name_links.append(row)

drug_code_class_conflicts = {
    code: ids for code, ids in class_ids_by_drug_code.items() if len(ids) > 1
}
class_drug_code_conflicts = {
    category_id: codes for category_id, codes in drug_codes_by_class_id.items() if len(codes) > 1
}
drug_name_differences = [
    row for row in comparable_drug_names if row["drug_name"] != row["category_name"]
]
category_link_status_counts = Counter(row["status"] for row in category_link_checks)
show_table(
    ["검사 항목", "결과"],
    [
        ["클래스 연결을 검사한 객체 정답 수", len(category_link_checks)],
        ["같은 JSON의 클래스 정의와 연결된 객체 정답 수", category_link_status_counts["일치"]],
        ["id가 없거나 정수가 아닌 categories 항목 수", len(invalid_local_category_rows)],
        ["category_id가 없거나 정수가 아닌 객체 정답 수", category_link_status_counts["category_id 누락·형식 확인"]],
        ["같은 JSON에서 클래스 ID를 찾지 못한 객체 정답 수", category_link_status_counts["같은 JSON에서 클래스 ID를 찾지 못함"]],
        ["같은 JSON에서 클래스 ID가 여러 항목과 연결된 객체 정답 수", category_link_status_counts["같은 JSON에서 클래스 ID가 여러 항목과 연결됨"]],
        ["이미지·클래스가 모두 하나로 연결된 객체 정답 수", len(drug_class_links)],
        ["약품 코드가 없거나 형식 확인이 필요한 연결 항목 수", len(missing_drug_code_links)],
        ["연결된 고유 약품 코드 수", len(class_ids_by_drug_code)],
        ["연결된 고유 클래스 ID 수", len(drug_codes_by_class_id)],
        ["여러 클래스 ID와 연결된 약품 코드 수", len(drug_code_class_conflicts)],
        ["여러 약품 코드와 연결된 클래스 ID 수", len(class_drug_code_conflicts)],
        ["약품명 원문 비교 대상 항목 수", len(comparable_drug_names)],
        ["약품명 결측·형식 문제로 비교하지 못한 항목 수", len(missing_drug_name_links)],
        ["약품명 원문 표기가 다른 항목 수", len(drug_name_differences)],
    ],
)

# 표기 차이·일대다 대응은 검토 대상으로 표시하고 원본 값은 보존한다.
for status, count in sorted(category_link_status_counts.items()):
    if status == "일치":
        continue
    display(Markdown(f"**{status}: 전체 {count}건, 최대 5건 표시**"))
    show_table(
        ["JSON 경로(data/raw 기준)", "annotation ID", "category_id"],
        [
            [row["json_path"], row["annotation"].get("id"), row["category_id"]]
            for row in category_link_checks if row["status"] == status
        ][:5],
    )
if invalid_local_category_rows:
    show_table(
        ["id 형식 확인이 필요한 JSON(최대 5건)", "categories.id"],
        [[row["json_path"], row.get("id")] for row in invalid_local_category_rows[:5]],
    )
if drug_code_class_conflicts:
    show_table(
        ["여러 클래스와 연결된 약품 코드(최대 5개)", "클래스 ID 목록"],
        [[key, sorted(values)] for key, values in sorted(drug_code_class_conflicts.items())[:5]],
    )
if class_drug_code_conflicts:
    show_table(
        ["여러 약품 코드와 연결된 클래스 ID(최대 5개)", "약품 코드 목록"],
        [[key, " / ".join(sorted(values))] for key, values in sorted(class_drug_code_conflicts.items())[:5]],
    )
for label, rows in [
    ("약품 코드 확인 대상", missing_drug_code_links),
    ("약품명 결측·형식 확인 대상", missing_drug_name_links),
    ("약품명 원문 표기 차이", drug_name_differences),
]:
    if rows:
        display(Markdown(f"**{label}: 전체 {len(rows)}건, 최대 5건 표시**"))
        show_table(
            ["JSON 경로(data/raw 기준)", "drug_N", "클래스 ID", "categories.name", "images.dl_name"],
            [[row["json_path"], row["drug_N"], row["category_id"], row["category_name"], row["drug_name"]] for row in rows[:5]],
        )

| 검사 항목 | 결과 |
| --- | --- |
| 클래스 연결을 검사한 객체 정답 수 | 763 |
| 같은 JSON의 클래스 정의와 연결된 객체 정답 수 | 763 |
| id가 없거나 정수가 아닌 categories 항목 수 | 0 |
| category_id가 없거나 정수가 아닌 객체 정답 수 | 0 |
| 같은 JSON에서 클래스 ID를 찾지 못한 객체 정답 수 | 0 |
| 같은 JSON에서 클래스 ID가 여러 항목과 연결된 객체 정답 수 | 0 |
| 이미지·클래스가 모두 하나로 연결된 객체 정답 수 | 763 |
| 약품 코드가 없거나 형식 확인이 필요한 연결 항목 수 | 0 |
| 연결된 고유 약품 코드 수 | 56 |
| 연결된 고유 클래스 ID 수 | 56 |
| 여러 클래스 ID와 연결된 약품 코드 수 | 0 |
| 여러 약품 코드와 연결된 클래스 ID 수 | 0 |
| 약품명 원문 비교 대상 항목 수 | 763 |
| 약품명 결측·형식 문제로 비교하지 못한 항목 수 | 0 |
| 약품명 원문 표기가 다른 항목 수 | 0 |

#### 2.3.4 약품 코드·클래스·약품명 대응표

**목적:** 56개 클래스별로 실제 객체 정답에서 연결된 약품 코드와 두 위치의 약품명을 나란히 확인한다.

순번은 확인용이며 학습용 인덱스가 아니다. 각 칸의 값이 여러 종류라면 모두 표시하고,
연결 기록이 없는 클래스도 목록에서 제외하지 않는다.

In [71]:
drug_class_catalog = [
    [
        number,
        " / ".join(sorted(drug_codes_by_class_id.get(category_id, set()))) or "연결된 코드 없음",
        category_id,
        " / ".join(sorted(names_by_class_id[category_id])),
        " / ".join(sorted(drug_names_by_class_id.get(category_id, set()))) or "연결된 약품명 없음",
    ]
    for number, category_id in enumerate(original_class_ids, start=1)
]
show_table(
    ["순번(확인용)", "약품 코드 drug_N", "원본 클래스 ID", "categories.name", "images.dl_name"],
    drug_class_catalog,
)

| 순번(확인용) | 약품 코드 drug_N | 원본 클래스 ID | categories.name | images.dl_name |
| --- | --- | --- | --- | --- |
| 1 | K-001900 | 1900 | 보령부스파정 5mg | 보령부스파정 5mg |
| 2 | K-002483 | 2483 | 뮤테란캡슐 100mg | 뮤테란캡슐 100mg |
| 3 | K-003351 | 3351 | 일양하이트린정 2mg | 일양하이트린정 2mg |
| 4 | K-003483 | 3483 | 기넥신에프정(은행엽엑스)(수출용) | 기넥신에프정(은행엽엑스)(수출용) |
| 5 | K-003544 | 3544 | 무코스타정(레바미피드)(비매품) | 무코스타정(레바미피드)(비매품) |
| 6 | K-003743 | 3743 | 알드린정 | 알드린정 |
| 7 | K-003832 | 3832 | 뉴로메드정(옥시라세탐) | 뉴로메드정(옥시라세탐) |
| 8 | K-004543 | 4543 | 에어탈정(아세클로페낙) | 에어탈정(아세클로페낙) |
| 9 | K-012081 | 12081 | 리렉스펜정 300mg/PTP | 리렉스펜정 300mg/PTP |
| 10 | K-012247 | 12247 | 아빌리파이정 10mg | 아빌리파이정 10mg |
| 11 | K-012778 | 12778 | 다보타민큐정 10mg/병 | 다보타민큐정 10mg/병 |
| 12 | K-013395 | 13395 | 써스펜8시간이알서방정 650mg | 써스펜8시간이알서방정 650mg |
| 13 | K-013900 | 13900 | 에빅사정(메만틴염산염)(비매품) | 에빅사정(메만틴염산염)(비매품) |
| 14 | K-016232 | 16232 | 리피토정 20mg | 리피토정 20mg |
| 15 | K-016262 | 16262 | 크레스토정 20mg | 크레스토정 20mg |
| 16 | K-016548 | 16548 | 가바토파정 100mg | 가바토파정 100mg |
| 17 | K-016551 | 16551 | 동아가바펜틴정 800mg | 동아가바펜틴정 800mg |
| 18 | K-016688 | 16688 | 오마코연질캡슐(오메가-3-산에틸에스테르90) | 오마코연질캡슐(오메가-3-산에틸에스테르90) |
| 19 | K-018147 | 18147 | 리리카캡슐 150mg | 리리카캡슐 150mg |
| 20 | K-018357 | 18357 | 종근당글리아티린연질캡슐(콜린알포세레이트)  | 종근당글리아티린연질캡슐(콜린알포세레이트)  |
| 21 | K-019232 | 19232 | 콜리네이트연질캡슐 400mg | 콜리네이트연질캡슐 400mg |
| 22 | K-019552 | 19552 | 트루비타정 60mg/병 | 트루비타정 60mg/병 |
| 23 | K-019607 | 19607 | 스토가정 10mg | 스토가정 10mg |
| 24 | K-019861 | 19861 | 노바스크정 5mg | 노바스크정 5mg |
| 25 | K-020014 | 20014 | 마도파정 | 마도파정 |
| 26 | K-020238 | 20238 | 플라빅스정 75mg | 플라빅스정 75mg |
| 27 | K-020877 | 20877 | 엑스포지정 5/160mg | 엑스포지정 5/160mg |
| 28 | K-021325 | 21325 | 아토르바정 10mg | 아토르바정 10mg |
| 29 | K-021771 | 21771 | 라비에트정 20mg | 라비에트정 20mg |
| 30 | K-022074 | 22074 | 리피로우정 20mg | 리피로우정 20mg |
| 31 | K-022347 | 22347 | 자누비아정 50mg | 자누비아정 50mg |
| 32 | K-022362 | 22362 | 맥시부펜이알정 300mg | 맥시부펜이알정 300mg |
| 33 | K-024850 | 24850 | 놀텍정 10mg | 놀텍정 10mg |
| 34 | K-025367 | 25367 | 자누메트정 50/850mg | 자누메트정 50/850mg |
| 35 | K-025438 | 25438 | 큐시드정 31.5mg/PTP | 큐시드정 31.5mg/PTP |
| 36 | K-025469 | 25469 | 아모잘탄정 5/100mg | 아모잘탄정 5/100mg |
| 37 | K-027733 | 27733 | 트윈스타정 40/5mg | 트윈스타정 40/5mg |
| 38 | K-027777 | 27777 | 카나브정 60mg | 카나브정 60mg |
| 39 | K-027926 | 27926 | 울트라셋이알서방정 | 울트라셋이알서방정 |
| 40 | K-027993 | 27993 | 졸로푸트정 100mg | 졸로푸트정 100mg |
| 41 | K-028763 | 28763 | 트라젠타정(리나글립틴) | 트라젠타정(리나글립틴) |
| 42 | K-029345 | 29345 | 비모보정 500/20mg | 비모보정 500/20mg |
| 43 | K-029451 | 29451 | 레일라정 | 레일라정 |
| 44 | K-029667 | 29667 | 리바로정 4mg | 리바로정 4mg |
| 45 | K-030308 | 30308 | 트라젠타듀오정 2.5/850mg | 트라젠타듀오정 2.5/850mg |
| 46 | K-031863 | 31863 | 아질렉트정(라사길린메실산염) | 아질렉트정(라사길린메실산염) |
| 47 | K-031885 | 31885 | 자누메트엑스알서방정 100/1000mg | 자누메트엑스알서방정 100/1000mg |
| 48 | K-032310 | 32310 | 글리아타민연질캡슐 | 글리아타민연질캡슐 |
| 49 | K-033009 | 33009 | 신바로정 | 신바로정 |
| 50 | K-033208 | 33208 | 에스원엠프정 20mg | 에스원엠프정 20mg |
| 51 | K-033880 | 33880 | 글리틴정(콜린알포세레이트) | 글리틴정(콜린알포세레이트) |
| 52 | K-034597 | 34597 | 제미메트서방정 50/1000mg | 제미메트서방정 50/1000mg |
| 53 | K-035206 | 35206 | 아토젯정 10/40mg | 아토젯정 10/40mg |
| 54 | K-036637 | 36637 | 로수젯정10/5밀리그램 | 로수젯정10/5밀리그램 |
| 55 | K-038162 | 38162 | 로수바미브정 10/20mg | 로수바미브정 10/20mg |
| 56 | K-041768 | 41768 | 카발린캡슐 25mg | 카발린캡슐 25mg |

#### 2.3.5 분석 결과와 필요한 조치

- **관찰한 사실:** 클래스는 56종이며 원본 ID는 1900~41768의 비연속 번호이다. 객체 정답 763개 모두 같은 JSON의 클래스 정의와 연결된다. 연결된 약품 코드 56개와 클래스 ID 56개는 일대일로 대응하며, 763개 항목에서 images.dl_name과 categories.name의 원문 표기 차이는 없다. 같은 클래스 ID의 이름 충돌도 없다.
- **학습에 미치는 영향:** 현재 데이터에서는 약품 코드·클래스·약품명의 대응을 일관되게 조회할 수 있다. 원본 클래스 ID는 연속 번호가 아니므로 연속 인덱스를 요구하는 모델에는 별도 대응표가 필요하다. 코드 대응이 일대일이라는 결과는 코드 문자열을 임의로 변환해도 된다는 의미는 아니다.
- **필요한 조치:** 검증한 약품 코드·클래스·약품명 대응을 보관하고, 모델 요구사항에 맞춰 원본 클래스 ID와 학습용 인덱스 0~55의 양방향 대응을 준비한다. 원본 코드와 이름은 유지하며 파일명만으로 정답 클래스를 만들지 않는다.

**후속 확인:** **3번**에서 실제 객체와 클래스의 대응을 관찰하고 **4번**에서 클래스별 이미지·객체 수를 분석한다. **6번**에서 데이터 보강의 필요성과 대상 클래스를 검토한다.

### 2.4 약제 외형·식별정보

**목적:** 약제 모양·색상·치수·식별 문자·분할선·마크의 정의와 실제 값을 확인하고,
유사한 약품을 구별하는 단서와 이미지 전처리에서 보존할 정보를 파악한다.

#### 2.4.1 외형·식별 필드의 정의·실제 값·자료형


In [79]:
show_field_group("appearance")

| key | 정의 | 실제 값·관찰 범위 | 자료형 |
| --- | --- | --- | --- |
| images.chart | 알약 제형 | 예: "이약은 양면볼록한 장방형의 흰색정제이다"; 고유 값 53개; null 18건 | NoneType, str |
| images.dl_custom_shape | 제조 모양 | "정제, 저작정", "장용정제", "서방형정제", "경질캡슐제", "연질캡슐제"; 고유 값 5개 | str |
| images.drug_shape | 알약 모양 | "장방형", "원형", "타원형", "팔각형", "육각형"; 고유 값 5개; null 18건 | NoneType, str |
| images.form_code_name | 정제 분류명 | "나정", "필름코팅정", "장용성필름코팅정", "서방성필름코팅정", "경질캡슐제, 산제", "연질캡슐제, 액상", "다층정"; 고유 값 7개; null 18건 | NoneType, str |
| images.thick | 알약 두께 | 관찰 범위: 0~9.8; 고유 값 45개 | float, int |
| images.leng_long | 알약 장축 | 관찰 범위: 0~24; 고유 값 47개 | float, int |
| images.leng_short | 알약 단축 | 관찰 범위: 0~12.58; 고유 값 44개 | float, int |
| images.print_front | 식별문자_앞 | 예: "BSP"; 고유 값 48개; 빈 문자열 "" 16건; null 18건 | NoneType, str |
| images.print_back | 식별문자_뒤 | 예: "5"; 고유 값 23개; 빈 문자열 "" 256건; null 18건 | NoneType, str |
| images.color_class1 | 색상 1 | "하양", "노랑", "갈색", "주황", "분홍", "초록", "연두", "노랑, 투명", "파랑"; 고유 값 9개; null 18건 | NoneType, str |
| images.color_class2 | 색상 2 | "초록", "하양", "파랑"; 고유 값 3개; 빈 문자열 "" 693건; null 18건 | NoneType, str |
| images.line_front | 알약 앞면 분할선 여부 | "-", "+"; 고유 값 2개; 빈 문자열 "" 552건; null 18건 | NoneType, str |
| images.line_back | 알약 뒷면 분할선 여부 | "-", "+"; 고유 값 2개; 빈 문자열 "" 630건; null 18건 | NoneType, str |
| images.mark_code_front_anal | 알약 앞면 마크 형태 | "kowa", "육각형", "pfizer", "집,화살,교회", "LG~,LG", "Hanmi"; 고유 값 6개; 빈 문자열 "" 642건; null 18건 | NoneType, str |
| images.mark_code_back_anal | 알약 뒷면 마크 형태 | 빈 문자열 "" 745건; null 18건 | NoneType, str |
| images.mark_code_front_img | 알약 앞면 마크 이미지 | "http://connectdi.com/design/img/drug/147938654295100047.jpg", "http://connectdi.com/design/img/drug/147938640332900169.jpg", "http://connectdi.com/design/img/drug/147938657883200003.jpg", "http://connectdi.com/design/img/drug/147938647484100129.jpg", "http://connectdi.com/design/img/drug/147938661567700181.jpg", "http://connectdi.com/design/img/drug/147938669511100018.jpg"; 고유 값 6개; 빈 문자열 "" 642건; null 18건 | NoneType, str |
| images.mark_code_back_img | 알약 뒷면 마크 이미지 | 빈 문자열 "" 745건; null 18건 | NoneType, str |
| images.mark_code_front | 알약 앞면 마크 코드 | 빈 문자열 "" 745건; null 18건 | NoneType, str |
| images.mark_code_back | 알약 뒷면 마크 코드 | 빈 문자열 "" 745건; null 18건 | NoneType, str |

#### 2.4.2 분석 결과와 필요한 조치

- **관찰한 사실:** 외형 설명·모양·색상·식별 문자·분할선·치수·마크 필드가 있으며 일부 값에는 null·빈 문자열이 있다. 치수는 int·float가 혼합되어 있고, 색상 1·2의 공통 부위 규칙과 치수 단위는 원본 설명만으로 확정되지 않는다.
- **학습에 미치는 영향:** 색상·각인·모양은 약품 구별에 도움이 될 수 있으나 메타데이터만으로 실제 이미지에서 보이는 특징을 확정할 수 없다. 색상 1·2가 같거나 비어 있다는 이유만으로 중복·오류로 판정할 수 없다.
- **필요한 조치:** null·빈 문자열을 임의의 색상이나 숫자 0으로 바꾸지 않고 구분한다. 두 색상 필드를 보존하며 실제 약제 이미지와 대조한다. 치수의 단위가 확인되기 전에는 박스 크기와 직접 비교하지 않는다.

**후속 확인:** **3번**에서 각인·색상·모양의 가시성을 관찰하고 **4·6번**에서 구별이 어려운 클래스와 전처리·증강·추가 데이터 후보를 검토한다.

### 2.5 촬영정보

**목적:** 약제 상태·배경·조명·방향·카메라 필드의 의미와 실제 값을 이해하고,
촬영 조건별 분석과 검증 설계에 활용할 수 있는 정보를 확인한다.

#### 2.5.1 촬영 필드의 정의·실제 값·자료형


In [80]:
show_field_group("capture")

| key | 정의 | 실제 값·관찰 범위 | 자료형 |
| --- | --- | --- | --- |
| images.drug_S | 알약상태 | "정상알약"로 동일 | str |
| images.back_color | 촬영배경 | "연회색 배경"로 동일 | str |
| images.light_color | 촬영조명 | "주백색"로 동일 | str |
| images.drug_dir | 알약방향 | "앞면", "뒷면"; 고유 값 2개 | str |
| images.camera_la | 카메라위도 | 70, 75, 90; 고유 값 3개 | int |
| images.camera_lo | 카메라경도 | 0로 동일 | int |

#### 2.5.2 촬영 필드의 제공 범위와 일관성

**목적:** 배경·조명·방향·카메라 관련 5개 필드가 같은 이미지에 대해 빠짐없이 일관되게 기록되어 있는지 확인한다.

`camera_la`·`camera_lo`의 공식 명칭은 카메라 위도·경도이지만 단위와 기준 좌표계는 확인되지 않았다.
이는 숫자가 기록되어 있지 않다는 뜻이 아니라, 숫자의 물리적 의미를 해석할 근거가 충분하지 않다는 뜻이다.

In [74]:
show_metadata_consistency({"back_color", "light_color", "drug_dir", "camera_la", "camera_lo"})

| key | 값 있는 images 항목 | 결측 항목 | 값 있는 고유 이미지 | 동일 이미지 내 값 불일치 수 |
| --- | --- | --- | --- | --- |
| images.back_color | 763/763 | 0 | 232 | 0 |
| images.light_color | 763/763 | 0 | 232 | 0 |
| images.drug_dir | 763/763 | 0 | 232 | 0 |
| images.camera_la | 763/763 | 0 | 232 | 0 |
| images.camera_lo | 763/763 | 0 | 232 | 0 |

#### 2.5.3 분석 결과와 필요한 조치

- **관찰한 사실:** 확인한 촬영 필드 5개는 images 항목 763개와 고유 이미지 232개에 제공되며 동일 이미지 내 값 불일치는 없다. 배경은 연회색·조명은 주백색 한 종류이고, drug_dir는 앞면·뒷면, camera_la는 70·75·90, camera_lo는 0으로 기록되어 있다.
- **학습에 미치는 영향:** 촬영 조건별 분석에 활용할 정보가 있으나 메타데이터 값만으로 실제 픽셀의 다양성이나 촬영 장면의 독립성을 확정할 수 없다. 관찰된 카메라 숫자를 곧바로 회전 각도나 좌표로 적용할 근거도 부족하다.
- **필요한 조치:** 실제 이미지와 촬영 메타데이터를 대조하고 숫자 필드의 단위·기준을 확인한다. 관찰된 배경·조명 조건의 범위를 바탕으로 증강·보강 후보와 촬영 그룹 분할 기준을 검토한다.

**집계 기준:** 결측은 필드 부재·null·빈 문자열 기준이며 고유 이미지 수는 파일명 기준이다. 위의 동일 이미지 내 일관성 검사는 명시한 5개 필드를 대상으로 한다.
**후속 확인:** **3·4번**에서 실제 촬영 조건과 분포를 확인하고 **6번**에서 검증 분할과 외부 데이터 필요성을 판단한다.

### 2.6 객체 정답·박스 정보

**목적:** 객체 식별자·박스·면적·평가 및 무시 관련 필드의 실제 값을 확인하고,
학습용 정답 형식으로 해석하기 전에 필요한 좌표·객체 검증 기준을 정한다.

#### 2.6.1 객체·박스 필드의 정의·실제 값·자료형


In [81]:
show_field_group("object")

| key | 정의 | 실제 값·관찰 범위 | 자료형 |
| --- | --- | --- | --- |
| annotations.id | 어노테이션 아이디 | 관찰 범위: 50~5691; 고유 값 763개; 식별자 번호 기준 | int |
| annotations.bbox | bbox 좌표 | 예: [644, 845, 189, 190]; 길이 4; 원소별 관찰 범위: [2~6567, 25~932, 125~529, 126~664] | list[int] |
| annotations.area | 바운딩박스 면적 | 관찰 범위: 23250~272435; 고유 값 635개 | int |
| annotations.iscrowd | 평가 분류 | 0로 동일 | int |
| annotations.ignore | 무시 여부 | 0로 동일 | int |
| annotations.segmentation | 라벨링 바운딩 박스 | []로 동일; 길이 0 | list |

#### 2.6.2 분석 결과와 필요한 조치

- **관찰한 사실:** bbox는 정수 4개로 구성되며 첫 번째 원소의 관찰 최댓값은 6567로 기록된 이미지 너비 976보다 크다. segmentation은 빈 목록으로 기록되어 있다. 박스 좌표의 원소 순서·기준 및 평가 관련 코드의 세부 의미는 추가 확인이 필요하다.
- **학습에 미치는 영향:** 원시 annotation 수 763개를 바로 유효 객체 수로 간주하거나 좌표를 임의 형식으로 변환하면 학습 정답이 잘못될 수 있다. 관찰 범위만으로 어느 좌표가 잘못되었는지 확정할 수 없다.
- **필요한 조치:** 실제 이미지에 박스를 그려 원소 순서와 좌표 기준을 확인하고, 이미지 경계·면적 관계·중복 객체를 검증한다. iscrowd·ignore 등은 사용할 학습기의 입력 규칙과 함께 해석한다.

**후속 확인:** **3번**에서 실제 객체와 박스의 대응을 시각화하고 **5번**에서 좌표·면적·중복과 image_id 연결의 유효성을 검사한다.

### 2.7 학습 준비를 위한 결론과 후속 확인

| 관찰한 사실 | 학습에 미치는 영향·필요한 조치 | 후속 확인 |
| --- | --- | --- |
| 전체 JSON의 key 구성은 같지만 일부 값에 null·빈 문자열·혼합 자료형이 있음 | key 존재와 값의 유효성을 구분하고 필드별 결측·변환 규칙을 정함 | 전처리 형식 결정 시 |
| 이미지 한 장에 JSON 2~4개가 연결되며 객체 763개의 이미지 ID 연결이 일치함. 이미지 ID·파일명은 232개씩 일대일 대응 | 이미지별 관련 JSON을 함께 조회하고 원본 ID 대응을 보관 | 3·5번: 실제 객체·박스·중복 확인 |
| 객체 763개의 클래스 참조가 일치하고 약품 코드·클래스 56개가 일대일 대응하며 약품명 차이는 없음. 원본 ID는 비연속 | 검증한 코드·이름 대응을 보관하고 모델에 맞는 원본 ID↔학습용 인덱스 대응표 준비 | 모델 입력 형식 결정 시 |
| 외형·색상·각인 정보가 있으나 결측과 의미가 미확정된 필드가 있음 | 실제 이미지에서 구별 단서를 확인하고 전처리·증강에서 보존할 특징을 검토 | 3·4·6번 |
| 배경·조명은 각각 한 종류로 기록됨 | 실제 촬영 조건의 다양성을 확인해 증강·보강 후보 검토 | 3·4·6번 |
| width·height와 별도로 size=200이 기록되며 단위·산정 방식은 미확인 | 실제 이미지 크기와 대조하고 size를 리사이즈 기준으로 사용하기 전 의미 확인 | 3·5번 |
| bbox 첫 원소의 관찰 최댓값이 기록된 너비보다 큼 | 좌표 형식을 확인하고 경계·면적·객체 대응을 검증한 뒤 유효 객체 수 확정 | 3·5번 |
| 동일 이미지의 정답이 여러 JSON에 나뉨 | 동일 이미지의 모든 정답을 같은 분할에 포함하고 유사 촬영 장면 관계도 검토 | 6번 |

**미확정 사항:** 파일명 숫자의 의미, 카메라 필드의 단위·좌표 기준,
size의 단위·산정 방식, 색상 1·2의 공통 부위 규칙, 치수 단위, bbox 유효성.